# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema).
2. A function or coroutine to execute.

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")

from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])


In [9]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the getWeather function for that. Let me check the function parameters. It requires a location, which in this case is Boston. So I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments. No other parameters are needed. Just the location. Alright, that should do it.\n", 'tool_calls': [{'id': 'r2end0xbj', 'function': {'arguments': '{"location":"Boston"}', 'name': 'getWeather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 153, 'total_tokens': 261, 'completion_time': 0.182895182, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.006267768, 'prompt_tokens_details': None, 'queue_time': 0.054473541, 'total_time': 0.18916295}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_re

## Tool Execution Loops


In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.


In [11]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the available tools. There\'s a function called getWeather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'pdv9r9a22', 'function': {'arguments': '{"location":"Boston"}', 'name': 'getWeather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 152, 'total_tokens': 245, 'completion_time': 0.140965677, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006986089, 'prompt_tokens_details': None, 'queue_time': 0.159348949, 'total_time': 0.147951766}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand'